# TensorFlow: U-Net Neural Network for Semantic Segmentation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay
from common.metric import get_confusion_matrix, get_iou, get_dice

In [ ]:
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '1'

import tensorflow as tf
import tensorflow_datasets as tfds

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Conv2DTranspose
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Concatenate

print('TF Version: ', tf.__version__)
print('TF Eager mode: ', tf.executing_eagerly())
print('TF GPU is", "available' if tf.config.list_physical_devices('GPU') else 'not available')

## General

In [ ]:
# The size of batch
BATCH_SIZE = 32
# THe number of epochs
EPOCHS = 50
# The size of sample image
IMAGE_SIZE = (128,128)
# THe shape of sample image
IMAGE_SHAPE = IMAGE_SIZE + (3,)
# The list of classes
CLASS_NAMES = ['pet', 'background', 'outline']

## Download Dataset

In [ ]:
datasets, info = tfds.load('oxford_iiit_pet', with_info=True)

In [ ]:
def preprocess_image(data):
    # Get image + label (annotation map)
    image = tf.image.resize(data['image'], IMAGE_SIZE, method='nearest')
    label = tf.image.resize(data['segmentation_mask'], IMAGE_SIZE, method='nearest')
    # Normalize
    image = tf.cast(image, dtype=tf.float32) / 255.0
    label = label - 1
    return image, label

In [ ]:
BUFFER_SIZE = 1000

tr_ds = (datasets['train']
    .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
    .cache()
    .shuffle(BUFFER_SIZE)
    .batch(BATCH_SIZE)
    .prefetch(buffer_size=tf.data.AUTOTUNE))

ts_ds = (datasets['test']
     .map(preprocess_image, num_parallel_calls=tf.data.AUTOTUNE)
     .cache()
     .batch(BATCH_SIZE)
     .prefetch(buffer_size=tf.data.AUTOTUNE))

## Define Model

In [ ]:
def conv2d_block(inputs, n_filters, kernel_size=(3,3)):
    x = inputs
    for i in range(2):
        x = Conv2D(filters=n_filters,
                   kernel_size=kernel_size,
                   kernel_initializer='he_normal',
                   padding='same')(x)
        x = Activation('relu')(x)
    return x

In [ ]:
def encoder_block(inputs, n_filters, pool_size=(2,2), dropout=0.3):
    fn = conv2d_block(inputs, n_filters=n_filters)
    x = MaxPooling2D(pool_size=pool_size)(fn)
    x = Dropout(dropout)(x)
    return fn, x

In [ ]:
def encoder(inputs):
    f1, p1 = encoder_block(inputs, n_filters=64)
    f2, p2 = encoder_block(p1, n_filters=128)
    f3, p3 = encoder_block(p2, n_filters=256)
    f4, p4 = encoder_block(p3, n_filters=512)
    return p4, (f1, f2, f3, f4)

In [ ]:
def bottleneck(inputs):
    return conv2d_block(inputs, n_filters=1024)

In [ ]:
def decoder_block(inputs, fn, n_filters, kernel_size=(3,3), strides=(2,2), dropout=0.3):
    un = Conv2DTranspose(n_filters, kernel_size=kernel_size, strides=strides, padding='same')(inputs)
    x = Concatenate()([un, fn])
    x = Dropout(dropout)(x)
    x = conv2d_block(x, n_filters)
    return x

In [ ]:
def decoder(inputs, fns, classes):
    f1, f2, f3, f4 = fns
    c6 = decoder_block(inputs, f4, n_filters=512)
    c7 = decoder_block(c6, f3, n_filters=256)
    c8 = decoder_block(c7, f2, n_filters=128)
    c9 = decoder_block(c8, f1, n_filters=64)
    outputs = tf.keras.layers.Conv2D(classes, (1,1), activation='softmax')(c9)
    return outputs

In [ ]:
def unet():
    inputs = Input(shape=IMAGE_SHAPE)
    x, fns = encoder(inputs)
    x = bottleneck(x)
    outputs = decoder(x, fns, classes=len(CLASS_NAMES))
    model = Model(inputs=inputs, outputs=outputs)
    return model

In [ ]:
model = unet()

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'])

In [ ]:
model.summary()

## Train Model

In [ ]:
TR_LENGTH = info.splits['train'].num_examples
VL_LENGTH = info.splits['test'].num_examples

TR_STEPS = TR_LENGTH // BATCH_SIZE
VL_STEPS = VL_LENGTH // BATCH_SIZE

In [ ]:
history = model.fit(
    tr_ds,
    epochs=EPOCHS,
    steps_per_epoch=TR_STEPS,
    validation_steps=VL_STEPS,
    validation_data=ts_ds)

## Evaluate Model

In [ ]:
n_classes = len(CLASS_NAMES)

In [ ]:
def get_images_and_labels():
    y_true_labels = []
    y_true_images = []
    ds = ts_ds.unbatch()
    ds = ds.batch(VL_LENGTH)
    for images, labels in ds.take(1):
        y_true_images = images.numpy()
        y_true_labels = labels.numpy()
    count = VL_LENGTH - (VL_LENGTH % BATCH_SIZE)
    y_true_labels = y_true_labels[:count]
    y_true_labels = np.argmax(y_true_labels, axis=3)
    return y_true_images[:count], y_true_labels

In [ ]:
# Get true images and labels from validation dataset
_, y_true_labels = get_images_and_labels()

In [ ]:
# Get predicted  labels using validation dataset
y_pred_labels = model.predict(ts_ds, steps=info.splits['test'].num_examples // BATCH_SIZE)
y_pred_labels = np.argmax(y_pred_labels, axis=3)

In [ ]:
# Calculate global confusion matrix over all samples
cms = []
gcm = np.full(shape=(n_classes, n_classes), fill_value=0, dtype=np.int64)
for (y_pred, y_true) in zip(y_pred_labels, y_true_labels):
    cm = get_confusion_matrix(y_pred, y_true, n_classes)
    cms.append(cm)
    gcm += cm

In [ ]:
# Display confusion matrix of particular sample
sample = 0
ds = ConfusionMatrixDisplay(
    confusion_matrix=cms[sample],
    display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(8, 8))
ds.plot(colorbar=False, ax=ax)
plt.show()

In [ ]:
# Calculates class-wise IoU metric
IoU = get_iou(gcm)
# Calculates class-wise Dice metric
Dice = get_dice(gcm)

In [ ]:
# Display class-wise matrics
df = pd.DataFrame(
    np.stack([IoU, Dice], axis=1),
    index=CLASS_NAMES,
    columns=['IoU', 'Dice'])

df